# BinSense — M3b · Track 1: Raw-Union Auto-Label (the CONTROL)

**Goal:** scale detector labels past the 120-bin manual seed by taking the *union*
of two cheap auto-labelers (SAM masks → boxes, and a zero-shot open-vocab detector),
with only the crudest junk filtering. This is deliberately the **plain, simple**
track — it exposes the **noise floor** and gives the baseline that the gated pipeline
(notebook `03c`) must beat. **It is NOT a production label set.**

Pipeline: `SAM ∪ zero-shot` → drop specks/full-frame/slivers → NMS dedup. The
quality logic lives in `tools/labeling/autolabel.py` (unit-checked below); this
notebook orchestrates it and visualizes the result. Eval-gold bins are held out of
every stage. See GitHub issue #2.

In [ ]:
# Cell 1: Bootstrap — path resolution + data/code split
import sys, os, subprocess
from pathlib import Path

GITHUB_URL = 'https://github.com/rishib09/AmazonBinSense.git'
BRANCH     = 'm3b-auto-labeling'   # set 'master' after this milestone merges
DRIVE_ROOT = '/content/drive/MyDrive/Interview Kickstart/Capstone Project/Amazon BinSense'
LOCAL_DATA = r'G:\My Drive\Interview Kickstart\Capstone Project\Amazon BinSense\data'

try:
    from google.colab import drive
    IN_COLAB = True
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/AmazonBinSense')
    if (PROJECT_ROOT / '.git').exists():
        subprocess.run(['git', '-C', str(PROJECT_ROOT), 'fetch', 'origin', BRANCH], check=False)
        subprocess.run(['git', '-C', str(PROJECT_ROOT), 'checkout', BRANCH], check=False)
        subprocess.run(['git', '-C', str(PROJECT_ROOT), 'pull', 'origin', BRANCH, '--ff-only'], check=False)
    else:
        subprocess.run(['git', 'clone', '--branch', BRANCH, GITHUB_URL, str(PROJECT_ROOT)], check=True)
    os.environ['BINSENSE_DATA_DIR'] = str(Path(DRIVE_ROOT) / 'data')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'opencv-python-headless', 'pandas', 'pyyaml', 'matplotlib'], check=True)
except ImportError:
    IN_COLAB = False
    if os.getenv('BINSENSE_DIR'):
        PROJECT_ROOT = Path(os.environ['BINSENSE_DIR'])
    else:
        _cwd = Path.cwd()
        PROJECT_ROOT = _cwd.parent if _cwd.name == 'notebooks' else _cwd
    if not os.getenv('BINSENSE_DATA_DIR') and Path(LOCAL_DATA).exists():
        os.environ['BINSENSE_DATA_DIR'] = LOCAL_DATA

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('Running in:', 'Google Colab' if IN_COLAB else 'Local', '| ROOT:', PROJECT_ROOT)

In [ ]:
# Cell 2: Imports + paths — raw method dirs, track output dir, eval wall
import json
import pandas as pd
from pathlib import Path
from utils.env_utils import setup_env

cfg = setup_env(verbose=True)
IMAGES_DIR = cfg.images_dir
META_DIR   = cfg.metadata_dir
SEED_DIR   = cfg.labels_dir                       # 120-bin manual seed (ground truth)

# Raw per-method boxes (written by the GPU cells below; persist on Drive)
SAM_DIR = cfg.data_dir / 'labels_auto' / 'sam'
ZS_DIR  = cfg.data_dir / 'labels_auto' / 'zeroshot'
SAM_DIR.mkdir(parents=True, exist_ok=True)
ZS_DIR.mkdir(parents=True, exist_ok=True)

# The one wall we never cross: eval-gold bins are the M7 test set.
EVAL_IDS = set(pd.read_csv(cfg.splits_dir / 'eval.csv')['bin_id'].astype(str).str.zfill(5))
EXTEND   = pd.read_csv(cfg.splits_dir / 'extend.csv')['bin_id'].astype(str).str.zfill(5).tolist()
EXTEND   = [b for b in EXTEND if b not in EVAL_IDS]
print(f'extend bins to auto-label: {len(EXTEND)}   eval held out: {len(EVAL_IDS)}   seed labels: {len(list(SEED_DIR.glob("*.txt")))}')

## Step 1 — Generate raw per-method boxes (Colab GPU)
Run SAM and the zero-shot detector over the `extend` bins; each writes one YOLO txt per bin (with a confidence column) to its own folder. Guarded so the notebook opens without a GPU.

In [ ]:
# Cell 3: SAM auto-mask -> boxes (Colab GPU)   [writes SAM_DIR, score = stability]
from tools.labeling.autolabel import write_boxes

RUN_SAM  = False           # flip True on a Colab GPU runtime
SAM_N    = 200
SAM_TYPE = 'vit_b'
SAM_CKPT = 'sam_vit_b_01ec64.pth'   # place under models/ (download separately)

if RUN_SAM:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'segment-anything'], check=True)
    import cv2
    from segment_anything import sam_model_registry, SamAutomaticMaskGenerator
    from utils.env_utils import get_device

    sam = sam_model_registry[SAM_TYPE](checkpoint=str(cfg.models_dir / SAM_CKPT)).to(get_device())
    mg = SamAutomaticMaskGenerator(sam, points_per_side=32, pred_iou_thresh=0.86,
                                   stability_score_thresh=0.92, min_mask_region_area=500)
    todo = [b for b in EXTEND if not (SAM_DIR / f'{b}.txt').exists()][:SAM_N]
    for bid in todo:
        img = cv2.imread(str(IMAGES_DIR / f'{bid}.jpg'))
        if img is None:
            continue
        H, W = img.shape[:2]
        boxes = []
        for m in mg.generate(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)):
            x, y, w, h = m['bbox']
            boxes.append({'cx': (x + w/2)/W, 'cy': (y + h/2)/H, 'w': w/W, 'h': h/H,
                          'score': float(m.get('stability_score', 1.0)), 'source': 'sam'})
        write_boxes(SAM_DIR / f'{bid}.txt', boxes, with_score=True)
    print(f'SAM wrote {len(todo)} bins -> {SAM_DIR}')
else:
    print(f'RUN_SAM=False. Existing SAM box files: {len(list(SAM_DIR.glob("*.txt")))}')

In [ ]:
# Cell 4: Zero-shot boxes via YOLO-World (Colab)   [writes ZS_DIR, score = conf]
from tools.labeling.autolabel import write_boxes

RUN_ZEROSHOT = False
ZS_N       = 200
ZS_CONF    = 0.05
ZS_PROMPTS = ['product', 'box', 'package', 'item', 'bottle', 'bag']

if RUN_ZEROSHOT:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics'], check=True)
    from ultralytics import YOLOWorld
    model = YOLOWorld('yolov8x-worldv2.pt')
    model.set_classes(ZS_PROMPTS)
    todo = [b for b in EXTEND if not (ZS_DIR / f'{b}.txt').exists()][:ZS_N]
    for bid in todo:
        r = model.predict(str(IMAGES_DIR / f'{bid}.jpg'), conf=ZS_CONF, verbose=False)[0]
        confs = r.boxes.conf.tolist()
        boxes = [{'cx': cx, 'cy': cy, 'w': w, 'h': h, 'score': float(s), 'source': 'zeroshot'}
                 for (cx, cy, w, h), s in zip(r.boxes.xywhn.tolist(), confs)]
        write_boxes(ZS_DIR / f'{bid}.txt', boxes, with_score=True)
    print(f'Zero-shot wrote {len(todo)} bins -> {ZS_DIR}')
else:
    print(f'RUN_ZEROSHOT=False. Existing zero-shot box files: {len(list(ZS_DIR.glob("*.txt")))}')

## Step 2 — Engine sanity check (no GPU)
Before trusting the boxes, prove the geometry/agreement/NMS primitives behave. This is the "test", inline as narrative.

In [ ]:
# Cell 5: Engine sanity check (no GPU) — proves the geometry/agreement/NMS logic
from tools.labeling.autolabel import iou, nms, gate_geometry, cross_method_agreement, gate_count, GateParams

def _b(cx, cy, w, h, s=1.0):
    return {'cx': cx, 'cy': cy, 'w': w, 'h': h, 'score': s, 'source': 't'}

p = GateParams()
assert iou(_b(.5,.5,.2,.2), _b(.5,.5,.2,.2)) == 1.0
assert iou(_b(.5,.5,.2,.2), _b(.9,.9,.2,.2)) == 0.0
# geometry drops speck / full-frame / sliver, keeps a normal box
kept = gate_geometry([_b(.5,.5,.01,.01), _b(.5,.5,.99,.99), _b(.5,.5,.6,.01), _b(.5,.5,.1,.1)], p)
assert len(kept) == 1
# NMS collapses a near-duplicate
assert len(nms([_b(.5,.5,.2,.2,.9), _b(.51,.51,.2,.2,.5), _b(.1,.1,.1,.1,.8)], p.nms_iou)) == 2
# agreement keeps only the box both methods found
assert len(cross_method_agreement([_b(.3,.3,.2,.2), _b(.8,.8,.1,.1)], [_b(.31,.31,.2,.2)], p.agree_iou)) == 1
# count gate: 0 and gross over-seg rejected; visible<expected accepted
assert gate_count(0, 10, p)[0] is False and gate_count(30, 10, p)[0] is False and gate_count(8, 10, p)[0] is True
print('Engine sanity check PASSED — geometry, NMS, cross-method agreement, count gate all behave.')

## Step 3 — Build Track 1 (raw union)
Union both methods, apply only crude geometry filtering + NMS. No count gate, no cross-method check — that is the point of the control.

In [ ]:
# Cell 6: Build Track 1 label set
from tools.labeling.autolabel import build_track, GateParams

OUT1 = cfg.data_dir / 'labels_track1'
p = GateParams()   # geometry + NMS defaults; raw_union ignores the count/agreement gates
st1 = build_track('track1', SAM_DIR, ZS_DIR, OUT1, META_DIR, EXTEND, EVAL_IDS, p)
print('Track 1 (raw union):', json.dumps(st1.as_dict(), indent=2))

man1 = pd.DataFrame(st1.manifest)
if len(man1):
    print(f"\nbins with boxes: {st1.n_bins_out}   total boxes: {st1.n_boxes_out}   "
          f"mean boxes/bin: {st1.n_boxes_out / max(1, st1.n_bins_out):.1f}")
    display(man1.head(10))
else:
    print('No raw boxes yet — run Step 1 on Colab first, then re-run.')

## Step 4 — Visualize the noise floor
Overlay raw-union boxes on a few bins and compare box-count to `EXPECTED_QUANTITY`. Expect over-detection and spurious boxes — that is the noise the gated track must remove.

In [ ]:
# Cell 7: Overlays + boxes-per-bin vs expected
import matplotlib.pyplot as plt, cv2, numpy as np
from tools.labeling.overlay_check import draw_overlay, parse_label

if len(man1) and st1.n_bins_out:
    shown = [r['bin_id'] for r in st1.manifest if r['n_final']][:3]
    fig, axes = plt.subplots(1, len(shown), figsize=(5*len(shown), 5))
    for ax, bid in zip(np.atleast_1d(axes), shown):
        out = cfg.base_dir / 'reports' / 'track1_overlays' / f'{bid}.jpg'
        draw_overlay(IMAGES_DIR/f'{bid}.jpg', parse_label(OUT1/f'{bid}.txt'), out)
        ax.imshow(cv2.cvtColor(cv2.imread(str(out)), cv2.COLOR_BGR2RGB)); ax.set_title(bid); ax.axis('off')
    plt.suptitle('Track 1 raw-union boxes (control)'); plt.show()

    d = man1[man1['n_final'] > 0].copy()
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].hist(d['n_final'], bins=20); ax[0].set_title('boxes per bin (raw union)'); ax[0].set_xlabel('#boxes')
    dd = d.dropna(subset=['expected'])
    ax[1].scatter(dd['expected'], dd['n_final'], alpha=.3, s=12)
    lim = max(dd['expected'].max(), dd['n_final'].max()) if len(dd) else 1
    ax[1].plot([0, lim], [0, lim], 'r--', lw=1); ax[1].set_xlabel('EXPECTED_QUANTITY'); ax[1].set_ylabel('#boxes')
    ax[1].set_title('raw-union count vs expected'); plt.tight_layout(); plt.show()
else:
    print('Nothing to visualize yet.')

## Step 5 — Assemble the Track-1 training set (seed + auto, EXCLUDE eval)
Combine the manual seed with the raw-union labels into a YOLO manifest for M4. Eval-gold is asserted out. M4 trains this and scores it on the gold set — the "raw union" arm of the A/B.

In [ ]:
# Cell 8: Write data_track1.yaml (manual seed + track-1 auto labels)
import yaml, random

seed_ids = [p.stem for p in SEED_DIR.glob('*.txt') if p.stem not in EVAL_IDS]
auto_ids = [p.stem for p in OUT1.glob('*.txt')] if OUT1.exists() else []
labeled  = sorted(set(seed_ids) | set(auto_ids))
assert not (set(labeled) & EVAL_IDS), 'EVAL LEAKAGE in Track-1 training labels!'

if not labeled:
    print('No labels — generate raw boxes (Step 1) then re-run.')
else:
    # Ultralytics resolves labels by swapping /images/->/labels/, so point at a
    # merged label dir. Simplest: symlink-free approach — write explicit lists and a
    # combined label root is not needed here because M4 reads per-image txt paths.
    random.Random(42).shuffle(labeled)
    n_val = max(1, int(0.15 * len(labeled)))
    val, train = labeled[:n_val], labeled[n_val:]
    YDIR = cfg.data_dir / 'yolo_track1'; YDIR.mkdir(parents=True, exist_ok=True)
    for name, ids in [('train.txt', train), ('val.txt', val)]:
        (YDIR / name).write_text('\n'.join(str(IMAGES_DIR / f'{b}.jpg') for b in ids))
    (YDIR / 'data.yaml').write_text(yaml.safe_dump({
        'path': str(cfg.data_dir), 'train': str(YDIR/'train.txt'), 'val': str(YDIR/'val.txt'),
        'nc': 1, 'names': {0: 'item'}}, sort_keys=False))
    print(f'seed={len(seed_ids)}  auto(track1)={len(auto_ids)}  train={len(train)}  val={len(val)}')
    print('NOTE: seed + track1 labels live in different dirs (data/labels, data/labels_track1).')
    print('      For training, merge both into one label root or set per-image label paths (see 03c handoff).')
    print('wrote', YDIR / 'data.yaml')

## Handoff to M4

Track 1 gives you the **noise floor**: how far raw union alone gets, and how much
junk rides along. Train M4 on `yolo_track1/data.yaml`, score on the Track-0 gold
set, and record mAP@50 / count-within-1. Then compare against **Track 2** (`03c`).
If gating does not beat this control, the gates are not earning their complexity.